# 54 — Results freeze: the last steps after notebooks 48, 50, 52, 51, 53

Run after the five figure notebooks. It (1) writes the PNG and CSV copies of `q6_half_of_cohort_heatmap` beside its PDF, (2) copies the tables the text cites that live only in `thesis_figures_v2` into `Data_for_writing/csv/`, (3) completes and prunes `MANIFEST.md`, (4) builds `FINAL_2026-10-14/` (NUMBERS_I-V, csv/, Figures/, MANIFEST.md; CHANGES.md is kept).

In [1]:
import sys, pathlib, shutil
PILLAR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(PILLAR))
import matplotlib.pyplot as plt
from viz import config, thesis
from viz import thesis_latex_part2 as h2, thesis_v2_part5 as v5
from viz.thesis_latex_common import root
cfg = config.load()
ctx = thesis.context(cfg)
R = root(cfg)
print(R)

/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/LaTeX_v15


/home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/labeling_evaluation/viz/frames.py:544: UserWarning: rapidfuzz not installed — fuzzy company matching disabled. Only exact lookup table matches will be applied.
  canon = g._normalise_company(sraw)


## 1. PNG and CSV of the heatmap beside its PDF

In [2]:
n = 'q6_half_of_cohort_heatmap'
fig, data, chk = h2.draw(h2._item(n), ctx, h2.FIGS[n][0])
o = R / 'Figures/Part_II_design_space/II3_distribution_and_new_variety/auxiliary'
o.mkdir(parents=True, exist_ok=True)
fig.savefig(o / f'{n}.png', dpi=200)
data.to_csv(o / f'{n}.csv', index=False)
plt.close(fig)
print('checks (text over text, under 7 pt):', chk)

checks (text over text, under 7 pt): (0, 0)


## 2. Tables the text cites that are written only to thesis_figures_v2

In [3]:
CITED_V2 = ['q2_largest_filer_per_class.csv']
for name in CITED_V2:
    src = next(v5.out_root(cfg).rglob(name))
    shutil.copy2(src, R / 'Data_for_writing' / 'csv' / name)
    print('copied', src.relative_to(v5.out_root(cfg)))

copied V1_who_files/main/q2_largest_filer_per_class.csv


## 3. MANIFEST: add the files written outside the part exporters, drop lines whose file no longer exists

In [4]:
IV2 = 'Figures/Part_IV_design_space_evolution/IV2_attributes_and_drivers/main/'
HEAT = 'Figures/Part_II_design_space/II3_distribution_and_new_variety/auxiliary/q6_half_of_cohort_heatmap'
EXTRA = [(f'{IV2}q6_iv2_merge_cvt_lc.{e}', 'yes', 'CVT and Lift + Cruise designs of 2012-15 and 2020-23 by tilting sets and propulsors per design; outlines and centre paths') for e in ('pdf', 'png', 'csv')] + \
        [(f'{IV2}q6_iv2_split_tp.{e}', 'yes', 'Tilt Propulsor splitting into one and two or more tilting sets, per cohort; three patent drawings') for e in ('pdf', 'png', 'csv')] + \
        [(f'{IV2}q6_iv2_merge_cvt_lc_distance.csv', 'yes', 'Gower distance (34 dimensions) of CVT* and of all CVT to Lift + Cruise per cohort, with the permutation band of the change 2012-15 to 2020-23')] + \
        [(f'{HEAT}.png', 'no (leaves the text)', 'the heatmap as PNG'), (f'{HEAT}.csv', 'no (leaves the text)', 'data behind q6_half_of_cohort_heatmap (figure folder copy)')] + \
        [('Data_for_writing/csv/q2_largest_filer_per_class.csv', 'yes', 'largest assignee per architecture, all years and in one cohort')]
mf = R / 'Data_for_writing' / 'MANIFEST.md'
lines = mf.read_text(encoding='utf-8').splitlines()
have = {l.split('`')[1] for l in lines if l.startswith('- `')}
lines += [f'- `LaTeX_v15/{p}` · {u} · {w}' for p, u, w in EXTRA if f'LaTeX_v15/{p}' not in have]
kept, dropped = [], []
for l in lines:
    if l.startswith('- `LaTeX_v15/'):
        p = R / l.split('`')[1].split('/', 1)[1]
        if not p.exists():
            dropped.append(l.split('`')[1]); continue
    kept.append(l)
seen, out = set(), []
for l in kept:
    if l not in seen:
        seen.add(l); out.append(l)
mf.write_text('\n'.join(out) + '\n', encoding='utf-8')
print(len(out), 'lines;', len(dropped), 'dropped (file gone):', dropped)

106 lines; 0 dropped (file gone): []


## 4. FINAL_2026-10-14/: the files the dissertation reads

In [5]:
F = R.parent / 'FINAL_2026-10-14'
keep = (F / 'CHANGES.md').read_text(encoding='utf-8') if (F / 'CHANGES.md').exists() else None
if F.exists():
    shutil.rmtree(F)
F.mkdir()
for n in ('NUMBERS_I.md', 'NUMBERS_II.md', 'NUMBERS_III.md', 'NUMBERS_IV.md', 'NUMBERS_V.md', 'MANIFEST.md'):
    shutil.copy2(R / 'Data_for_writing' / n, F / n)
shutil.copytree(R / 'Data_for_writing' / 'csv', F / 'csv')
shutil.copytree(R / 'Figures', F / 'Figures')
if keep is not None:
    (F / 'CHANGES.md').write_text(keep, encoding='utf-8')
files = sorted(p.relative_to(F) for p in F.rglob('*') if p.is_file())
print(len(files), 'files in', F)

106 files in /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/FINAL_2026-10-14
